In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from cuml.manifold import UMAP


PATH_EMBEDDINGS = Path(...)
if not PATH_EMBEDDINGS.exists():
    raise FileNotFoundError(f"Could not find `{PATH_EMBEDDINGS}`")
print(f"Set embedding to `{PATH_EMBEDDINGS}`")

PATH_LOOKUP = Path(...)
if not PATH_LOOKUP.exists():
    raise FileNotFoundError(f"Could not find `{PATH_LOOKUP}`")
print(f"Set track lookup to `{PATH_LOOKUP}`")

PATH_OUTPUT_DIR = Path(...)
PATH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Set output directory to `{PATH_OUTPUT_DIR}`.")

In [ ]:
print("Loading embeddings ...")
embs_df = pd.read_parquet(PATH_EMBEDDINGS)
emb_cols = [c for c in embs_df.columns if c.startswith("e") and c[1:].isdigit()]
print(f"{len(embs_df):,} tracks × {len(emb_cols)} dims")

print("Loading lookup ...")
lookup_df = pd.read_parquet(PATH_LOOKUP)
print(f"{len(lookup_df):,} tracks × {len(lookup_df.columns)} dims")

# Track embeddings

In [1]:
nc = 2
nn = 50
md = 0.01
metric = "cosine"

In [ ]:
matrix = embs_df[emb_cols].to_numpy(dtype=np.float32)
reducer = UMAP(
    n_components=nc,
    n_neighbors=nn,
    min_dist=md,
    metric=metric,
    verbose=False,
)
coords = reducer.fit_transform(matrix)

## Exporting

In [ ]:
run_tag = PATH_EMBEDDINGS.stem
if run_tag.startswith("embeddings_"):
    run_tag = run_tag[len("embeddings_"):]
md_str = str(md).replace(".", "d")

out_path = PATH_OUTPUT_DIR / f"umap_track_{nc}d_{run_tag}_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "track_rowid": embs_df["track_rowid"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

# Artists, albums, labels

In [ ]:
from src.topo import album_embeddings


album_df = album_embeddings(embs_df, lookup_df, min_tracks=5)
coords = reducer.transform(album_df[emb_cols].to_numpy(dtype=np.float32))

out_path = PATH_OUTPUT_DIR / f"umap_album_{nc}d_{run_tag}_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "album_rowid": album_df["album_rowid"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

In [ ]:
from src.topo import artist_embeddings


artist_df = artist_embeddings(embs_df, lookup_df, min_tracks=10)
coords = reducer.transform(artist_df[emb_cols].to_numpy(dtype=np.float32))

out_path = PATH_OUTPUT_DIR / f"umap_artist_{nc}d_{run_tag}_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "artist_rowid": artist_df["artist_rowid"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

In [ ]:
from src.topo import label_embeddings


label_df = label_embeddings(embs_df, lookup_df, min_tracks=100)
coords = reducer.transform(label_df[emb_cols].to_numpy(dtype=np.float32))

out_path = PATH_OUTPUT_DIR / f"umap_label_{nc}d_{run_tag}_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "label": label_df["label"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

# Visualize

In [ ]:
BINS = 1000

track_df = pd.read_parquet(PATH_OUTPUT_DIR / f"umap_track_{nc}d_{run_tag}_nn{nn}_md{md_str}_{metric}.parquet")
x = track_df["umap_x"].to_numpy()
y = track_df["umap_y"].to_numpy()

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

xi = np.clip(((x - x_min) / (x_max - x_min) * BINS).astype(np.int32), 0, BINS - 1)
yi = np.clip(((y - y_min) / (y_max - y_min) * BINS).astype(np.int32), 0, BINS - 1)

heatmap = np.bincount(yi * BINS + xi, minlength=BINS * BINS).reshape(BINS, BINS)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm


with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(
        heatmap,
        origin="lower",
        norm=PowerNorm(gamma=0.3, vmin=0., vmax=heatmap.max()),
        cmap="inferno",
        interpolation="nearest",
        extent=[x_min, x_max, y_min, y_max],
        aspect="auto",
    )
    fig.colorbar(im, ax=ax, fraction=0.046, shrink=0.3, pad=0.04)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()